In [21]:
"""
Combined Feature Elimination: Domain Knowledge + Smart VIF
===========================================================
Two-phase approach:
 
  Phase 1 — Domain-driven collapse
 
  Phase 2 — Smart VIF elimination
 
Result: 30 features → 8 features, AUC loss < 0.01.
 
"""
 
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.svm import SVC
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
DATA_PATH     = "Data for Task 1.csv"
TARGET_COL    = "diagnosis"
TARGET_POS    = "M"
DROP_COLS     = ["id", "Unnamed: 32"]
VIF_THRESHOLD = 10
STOP_N_FEATS  = 8     
CV_FOLDS      = 5
RANDOM_STATE  = 42

In [3]:
# Phase 1: Domain-driven drops
 
DOMAIN_DROPS = [
    "perimeter_mean",    
    "area_mean",        
    "perimeter_worst",  
    "area_worst",       
    "perimeter_se",      
    "concavity_mean",   
    "concavity_worst",  
    "texture_mean", 
]

In [4]:
# Create X and y first
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

y = (df[TARGET_COL] == TARGET_POS).astype(int)
X = df.drop(columns=[TARGET_COL])

target_corr = X.corrwith(y).abs()

domain_drops = DOMAIN_DROPS
vif_threshold = VIF_THRESHOLD
stop_n_feats = STOP_N_FEATS

In [5]:
def compute_vif(df: pd.DataFrame) -> dict[str, float]:
    arr = df.values.astype(float)
    return {
        col: variance_inflation_factor(arr, i)
        for i, col in enumerate(df.columns)
    }
 
 
def compute_auc(X: pd.DataFrame, y: pd.Series) -> float:
    sc  = StandardScaler()
    Xs  = sc.fit_transform(X)
    lr  = LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)
    return float(cross_val_score(lr, Xs, y, cv=CV_FOLDS, scoring="roc_auc").mean())

In [6]:
def run(
    data_path:     str   = DATA_PATH,
    target_col:    str   = TARGET_COL,
    target_pos:    str   = TARGET_POS,
    drop_cols:     list  = DROP_COLS,
    domain_drops:  list  = DOMAIN_DROPS,
    vif_threshold: float = VIF_THRESHOLD,
    stop_n_feats:  int   = STOP_N_FEATS,
) -> tuple[list[str], pd.DataFrame]:

    df = pd.read_csv(DATA_PATH)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    y = (df[TARGET_COL] == TARGET_POS).astype(int)
    X = df.drop(columns=[TARGET_COL])

    target_corr = X.corrwith(y).abs()
    
    final_features = list(current.columns)
    log = pd.DataFrame(log_rows)

    return final_features, log
    

In [7]:
 # Phase 1
print("=" * 72)
print("PHASE 1 — Domain-driven elimination")
print("=" * 72)


auc_before = compute_auc(X, y)
X_p1 = X.drop(columns=[c for c in domain_drops if c in X.columns])
auc_after  = compute_auc(X_p1, y)
 
for feat in domain_drops:
 
        print(f"\n  {len(X.columns)} → {len(X_p1.columns)} features  |  "
          f"AUC {auc_before:.4f} → {auc_after:.4f}")
 
    # Phase 2
print("\n" + "=" * 72)
print("PHASE 2 — Smart VIF elimination")
print(f"  Drop highest-VIF feature with LOWEST target correlation at each step")
print(f"  Stop when: n_features = {stop_n_feats} or all VIF ≤ {vif_threshold}")
print("=" * 72)
print(f"\n{'Step':<5} {'Features':>8} {'Dropped':<32} "
          f"{'VIF':>9} {'Target r':>10} {'AUC':>8}")
print("-" * 76)
print(f"{'0':<5} {len(X_p1.columns):>8} {'— after phase 1 —':<32} "
          f"{'':>9} {'':>10} {auc_after:>8.4f}")
 
log_rows = [{
        "phase": 1, "step": 0,
        "n_features": len(X_p1.columns),
        "feature_dropped": None, "vif_of_dropped": None,
        "target_r_of_dropped": None, "auc": round(auc_after, 4),
    }]
 
current = X_p1.copy()
step    = 0
 
while True:
        if stop_n_feats and len(current.columns) <= stop_n_feats:
            break
 
        vif      = compute_vif(current)
        high_vif = {col: v for col, v in vif.items() if v > vif_threshold}
 
        if not high_vif:
            print(f"\n  All VIFs ≤ {vif_threshold} — stopping naturally.")
            break
 
        drop    = min(high_vif.keys(), key=lambda f: target_corr[f])
        step   += 1
        current = current.drop(columns=[drop])
        auc     = compute_auc(current, y)
 
log_rows.append({
            "phase": 2, "step": step,
            "n_features": len(current.columns),
            "feature_dropped": drop,
            "vif_of_dropped": round(high_vif[drop], 1),
            "target_r_of_dropped": round(target_corr[drop], 4),
            "auc": round(auc, 4),
        })
print(f"{step:<5} {len(current.columns):>8} {drop:<32} "
              f"{high_vif[drop]:>9,.1f} {target_corr[drop]:>10.4f} {auc:>8.4f}")

PHASE 1 — Domain-driven elimination

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

  30 → 22 features  |  AUC 0.9953 → 0.9956

PHASE 2 — Smart VIF elimination
  Drop highest-VIF feature with LOWEST target correlation at each step
  Stop when: n_features = 8 or all VIF ≤ 10

Step  Features Dropped                                VIF   Target r      AUC
----------------------------------------------------------------------------
0           22 — after phase 1 —                                       0.9956
14           8 area_se                               19.8     0.5482   0.9906


In [8]:
# Summary
final_features = list(current.columns)
final_vif      = compute_vif(current)
 
print("\n" + "=" * 72)
print(f"Final feature set — {len(final_features)} features\n")
print(f"  {'Feature':<35} {'VIF':>8}  {'Target r':>10}")
print("  " + "-" * 56)
for f in sorted(final_features, key=lambda f: -target_corr[f]):
        flag = "  ← still high" if final_vif[f] > vif_threshold else ""
        print(f"  {f:<35} {final_vif[f]:>8.2f}  {target_corr[f]:>10.4f}{flag}")
 
log = pd.DataFrame(log_rows)
print(f"\n  AUC — all {len(X.columns)} features : {auc_before:.4f}")
print(f"  AUC — {len(final_features)} features        : {log.iloc[-1]['auc']:.4f}")
print(f"  AUC loss              : {auc_before - log.iloc[-1]['auc']:.4f}")




Final feature set — 8 features

  Feature                                  VIF    Target r
  --------------------------------------------------------
  concave points_worst                   54.80      0.7936  ← still high
  concave points_mean                    26.09      0.7766  ← still high
  radius_worst                          361.75      0.7765  ← still high
  radius_mean                           281.67      0.7300  ← still high
  compactness_mean                       41.05      0.5965  ← still high
  compactness_worst                      28.28      0.5910  ← still high
  radius_se                              10.91      0.5671  ← still high
  concavity_se                            3.67      0.2537

  AUC — all 30 features : 0.9953
  AUC — 8 features        : 0.9906
  AUC loss              : 0.0047


In [9]:
if __name__ == "__main__":
    final_features, log = run()
 
    log.to_csv("combined_elimination_log.csv", index=False)
    print("\nElimination log  → combined_elimination_log.csv")
 
    df = pd.read_csv(DATA_PATH)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    df[[TARGET_COL] + final_features].to_csv("data_combined_reduced.csv", index=False)
    print("Reduced dataset  → data_combined_reduced.csv")


Elimination log  → combined_elimination_log.csv
Reduced dataset  → data_combined_reduced.csv


In [12]:
df_reduced = pd.read_csv("data_combined_reduced.csv")
df_reduced = df_reduced.loc[:, ~df_reduced.columns.str.contains("^Unnamed")]

y = df_reduced["diagnosis"]
X = df_reduced.drop(columns=["diagnosis"])



In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

lr = LogisticRegression(
    max_iter=10000,
    random_state=42,
    class_weight="balanced"
)

lr.fit(X_train_sc, y_train)

y_pred = lr.predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred)

cv_f1 = cross_val_score(
    lr,
    X_train_sc,
    y_train,
    cv=cv,
    scoring="f1_macro"
).mean()

In [22]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_model(name, model, Xtr, Xte, use_sample_weight=False):

    if use_sample_weight:
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
        model.fit(Xtr, y_train, sample_weight=sample_weight)
    else:
        model.fit(Xtr, y_train)

    y_pred = model.predict(Xte)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xte)[:, 1]
    else:
        y_prob = model.decision_function(Xte)

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    cv_scores = []

    for train_idx, val_idx in cv.split(Xtr, y_train):
        model_cv = clone(model)

        if isinstance(Xtr, np.ndarray):
            X_fold_train = Xtr[train_idx]
            X_fold_val = Xtr[val_idx]
        else:
            X_fold_train = Xtr.iloc[train_idx]
            X_fold_val = Xtr.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        if use_sample_weight:
            fold_weight = compute_sample_weight(
                class_weight="balanced",
                y=y_fold_train
            )
            model_cv.fit(X_fold_train, y_fold_train, sample_weight=fold_weight)
        else:
            model_cv.fit(X_fold_train, y_fold_train)

        y_fold_pred = model_cv.predict(X_fold_val)
        cv_scores.append(f1_score(y_fold_val, y_fold_pred, pos_label="M"))

    cv_scores = np.array(cv_scores)

    return {
        "name": name,
        "model": model,
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "accuracy": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score((y_test == "M").astype(int), y_prob),
        "cv_f1": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "fn": cm[1, 0],
        "fp": cm[0, 1],
        "cm": cm,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

models_to_run = {
    "Logistic Regression (balanced, 21 features)": (
        LogisticRegression(
            max_iter=10000,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),

    "Shallow Decision Tree (balanced, 21 features)": (
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),

"Explainable Boosting Machine (balanced, 21 features)": (
    ExplainableBoostingClassifier(
        interactions=0,
        outer_bags=8,
        max_rounds=300,
        random_state=42
    ),
    X_train,
    X_test,
    True
),

    "Linear SVM (balanced, 21 features)": (
        SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    )
}


results_21_balanced = {}

print(f"\n  {'Model':<55} {'F1':>7} {'Recall':>7} {'Prec':>7} {'AUC':>7} {'CV F1':>8} {'FN':>4} {'FP':>4}")
print("  " + "-" * 110)

for name, (model, Xtr, Xte, use_sample_weight) in models_to_run.items():
    result = evaluate_model(name, model, Xtr, Xte, use_sample_weight)
    results_21_balanced[name] = result

    flag = " ✓" if result["f1"] > 0.95 else ""

    print(
        f"  {name:<55} "
        f"{result['f1']:>7.4f} "
        f"{result['recall']:>7.4f} "
        f"{result['precision']:>7.4f} "
        f"{result['auc']:>7.4f} "
        f"{result['cv_f1']:>8.4f} "
        f"{result['fn']:>4} "
        f"{result['fp']:>4}"
        f"{flag}"
    )



  Model                                                        F1  Recall    Prec     AUC    CV F1   FN   FP
  --------------------------------------------------------------------------------------------------------------
  Logistic Regression (balanced, 21 features)              0.9647  0.9762  0.9535  0.9983   0.9217    1    2 ✓
  Shallow Decision Tree (balanced, 21 features)            0.9398  0.9286  0.9512  0.9742   0.8856    3    2
  Explainable Boosting Machine (balanced, 21 features)     0.9512  0.9286  0.9750  0.9960   0.9265    3    1 ✓
  Linear SVM (balanced, 21 features)                       0.9655  1.0000  0.9333  0.9987   0.9216    0    3 ✓
